# Stable Diffusion Fine-Tuning — Сарны гадаргуу дээр сансрын хувцастай хүний зураг

**Сэдэв:** Diffusion загварыг өөрийн зургийн датасет дээр fine-tuning хийх

**Зорилго:** `runwayml/stable-diffusion-v1-5` загварыг өөрийн зургаар fine-tuning хийж,
img2img pipeline-аар саран дээр сансрын хувцастай зураг үүсгэх.

| Алхам | Тайлбар |
|-------|----------|
| 1 | Сангууд суулгах, суурь загвар туршиж харах |
| 2 | Өөрийн датасет бэлтгэх, Qwen2-VL caption үүсгэх |
| 3 | Загварыг fine-tuning хийх |
| 4 | Fine-tuned загвараас дүрс авах, img2img-ээр өөрийн зург оруулах |

> **Яагаад SD1.5?** SD3 нь T4 GPU дээр ~10GB+ VRAM шаарддаг тул OOM алдаа гардаг.
> SD1.5 нь ~1.7GB VRAM ашиглаж, img2img-г бүрэн дэмждэг, train_text_to_image.py-тай нийцдэг.


## 1-р алхам: Суурь загварыг бэлтгэх ба туршиж харах

### 1а. Шаардлагатай сангуудыг суулгах

| Сан | Зориулалт |
|-----|-----------|
| `diffusers` | Diffusion загвар ачаалах, ажиллуулах |
| `accelerate` | Mixed precision сургалт |
| `datasets` | HuggingFace датасет ачаалах |
| `bitsandbytes` | 8-bit Adam optimizer (санах ой хэмнэх) |


In [ ]:
%%capture
!pip install git+https://github.com/huggingface/diffusers.git
!pip install accelerate datasets bitsandbytes transformers sentencepiece
!pip install qwen-vl-utils  # Qwen2-VL caption үүсгэхэд


### 1б. Загварын нэрийг тохируулах

SD1.5 нь `train_text_to_image.py` скрипттэй бүрэн нийцдэг, T4-д тохиромжтой.


In [ ]:
%env MODEL_NAME=runwayml/stable-diffusion-v1-5


### 1в. Зургийг харуулах туслах функц


In [ ]:
import matplotlib.pyplot as plt

def plot_images(images, titles=None, figsize=(20, 6)):
    """
    Зургийн жагсаалтыг нэг мөрөнд харуулах.
    Args:
        images: PIL Image-уудын жагсаалт
        titles: Зураг бүрийн гарчиг (заавал биш)
        figsize: Зургийн хэмжээ
    """
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]

    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(titles[i] if titles else f"Зураг {i+1}", fontsize=10)

    plt.tight_layout()
    plt.show()
    print(f"✅ {n} зураг харуулагдлаа")


### 1г. HuggingFace нэвтрэх

> **Аюулгүй байдал:** Colab Secrets-д `HF_TOKEN` нэмэх: `Edit → Notebook settings → Secrets`.


In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("✅ HuggingFace-д амжилттай нэвтэрлээ.")
    else:
        print("⚠️  HF_TOKEN secret хоосон байна. Colab Secrets-д нэм.")
except Exception as e:
    print(f"⚠️  HF_TOKEN олдсонгүй: {e}")
    print("   login(token='YOUR_TOKEN') гэж гараар нэвтэрч болно.")


### 1д. Шаардлагатай сангуудыг импортлох


In [ ]:
import os
import torch
import gc

from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline

print(f"✅ PyTorch хувилбар: {torch.__version__}")
print(f"   GPU боломжтой эсэх: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU нэр: {torch.cuda.get_device_name(0)}")
    print(f"   GPU санах ой: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


### 1е. Суурь загварыг ачаалах

`StableDiffusionPipeline` (SD1.5) нь T4 GPU дээр ~1.7GB VRAM ашиглана.

- `torch_dtype=torch.float16` → GPU санах ойн хэрэглээг хагасаар бууруулна
- `safety_checker=None` → NSFW шүүлтийг хасаж хурдасгана


In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# ✅ SD1.5 — T4-д тохиромжтой, img2img дэмждэг
MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,          # Хурдасгахын тулд
    requires_safety_checker=False,
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()   # VRAM хэмнэх

print("✅ Суурь загвар амжилттай ачаалагдлаа!")
print(f"   VRAM ашиглалт: {torch.cuda.memory_allocated()/1e9:.1f} GB")


### 1ж. Суурь загвараас жишээ зураг үүсгэх

Fine-tuning хийхийн өмнө суурь загвар ямар зураг гаргадгийг харна.
Эдгээрийг 4-р алхамд fine-tuned загварын зурагтай харьцуулна.


In [ ]:
import torch

# Сарны гадаргуу дээрх сансрын хувцастай хүний prompt
BASE_PROMPT = (
    "astronaut in white NASA spacesuit standing on lunar surface, "
    "grey moon regolith ground, Earth visible in black starry sky background, "
    "cinematic 4k, sharp focus, photorealistic, dramatic lighting"
)
NEGATIVE_PROMPT = (
    "blurry, low quality, deformed, extra limbs, bad anatomy, "
    "text, watermark, cartoon, painting, illustration"
)

print(f"🎨 Prompt: '{BASE_PROMPT}'")
print("⏳ Суурь загвараас 4 зураг үүсгэж байна...")

base_images = []

for i in range(4):
    print(f"  {i+1}/4 үүсгэж байна...")
    torch.cuda.empty_cache()

    with torch.no_grad():
        image = pipe(
            BASE_PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=30,     # ✅ 8 → 30: чанар мэдэгдэхүйц сайжрана
            guidance_scale=9.0,         # ✅ 4.5 → 9.0: prompt-г чанд дагах
            height=512,
            width=512,
        ).images[0]
    base_images.append(image)

print(f"\n✅ {len(base_images)} зураг үүслээ!")


In [ ]:
print("🖼️  Суурь загвараас гарсан зургууд (Fine-tuning хийхийн өмнө):")
plot_images(base_images, titles=[f"Суурь #{i+1}" for i in range(len(base_images))])


### 1з. GPU санах ойг чөлөөлөх

Fine-tuning эхлүүлэхийн өмнө бүх объектыг устган санах ойг чөлөөлнө.


In [ ]:
print("🧹 GPU санах ойг цэвэрлэж байна...")

# ✅ Засвар: base_images-г устгах (өмнө нь 'images' гэж буруу байсан)
del pipe
del base_images
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    total_mem = torch.cuda.mem_get_info()[1] / 1e9
    print(f"✅ GPU санах ой чөлөөлөгдлөө: {free_mem:.1f}/{total_mem:.1f} GB чөлөөтэй")


---
## 2-р алхам: Өөрийн датасет бэлтгэх

### Датасетийн шаардлага

| Шаардлага | Тайлбар |
|-----------|---------|
| Зургийн тоо | 50–100 зураг хангалттай |
| Хэмжээ | 512×512 эсвэл том → crop |
| Агуулга | Өөрийн нүүр, биеийн зураг |
| Байрлал | `/content/my_images/` хавтасанд |

### 2а. Зургуудаа Colab-д upload хийх


In [ ]:
import os
from pathlib import Path

# Зургийн хавтас үүсгэх
IMAGE_DIR = "/content/my_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

# Colab-д файл upload хийх
from google.colab import files

print("📂 Зургуудаа сонгоно уу (jpg/png/webp):")
uploaded = files.upload()

for filename, data in uploaded.items():
    filepath = os.path.join(IMAGE_DIR, filename)
    with open(filepath, 'wb') as f:
        f.write(data)
    print(f"  ✅ {filename} хадгалагдлаа")

img_count = len(list(Path(IMAGE_DIR).glob("*.jpg")) +
               list(Path(IMAGE_DIR).glob("*.png")) +
               list(Path(IMAGE_DIR).glob("*.webp")))
print(f"\n📊 Нийт зургийн тоо: {img_count}")


### 2б. Автомат Caption Үүсгэгч (Qwen2-VL-7B)

Зураг тус бүрт Qwen2-VL загварыг ашиглан нарийвчилсан caption автоматаар үүсгэнэ.

> **Шаардлага:** `/workspace/Qwen2-VL-7B` загвар урьдчилан татагдсан байх ёстой.


In [ ]:
# ── Автомат Caption Үүсгэгч — Qwen2-VL-7B ──────────────────────────────
#
# Зураг бүрт автоматаар caption үүсгэж JSON болон txt файлуудад хадгална.
#
# Гаралт:
#   captions.json  → {"image1.jpg": "caption text ...", ...}
#   metadata.jsonl → HuggingFace датасетийн формат
#   train/         → зураг + тохирох .txt caption файлууд

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import os, json, shutil
from tqdm import tqdm
from pathlib import Path
import torch


class ImageCaptioner:
    """
    Qwen2-VL загвар ашиглан зураг бүрт автоматаар caption үүсгэдэг класс.

    Хэрэглэх:
        captioner = ImageCaptioner()
        caption = captioner.generate_caption("path/to/image.jpg")
    """

    def __init__(self, model_path="/workspace/Qwen2-VL-7B"):
        print(f"⬇️  Загвар ачааллаж байна: {model_path}")

        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_path,
            torch_dtype="auto",
            device_map="auto"
        )

        min_pixels = 256 * 28 * 28
        max_pixels = 1280 * 28 * 28
        self.processor = AutoProcessor.from_pretrained(
            model_path,
            min_pixels=min_pixels,
            max_pixels=max_pixels
        )
        print("✅ ImageCaptioner бэлэн боллоо!")

    def generate_caption(self, image_path: str) -> str | None:
        """Нэг зурагт нарийвчилсан caption үүсгэх."""
        try:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path},
                        {
                            "type": "text",
                            # ✅ Сайжруулсан prompt: нүүрний онцлог, хувцас, гэрэлтүүлэгт анхаарна
                            "text": """Please provide a detailed description of this image for AI training, including:
1. Main subject and their appearance. If a woman/female/adult/individual is present, describe as ***asian woman***.
2. Facial features: eye shape, skin tone, hair color and style.
3. Pose, expression, and body language.
4. Clothing and accessories in detail (color, style, material).
5. Background and setting (location, environment).
6. Lighting style (natural, studio, dramatic) and atmosphere.
7. Camera angle and composition (close-up, full body, etc.).
Write as a single flowing paragraph suitable for image generation prompts."""
                        },
                    ],
                }
            ]

            text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = self.processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to("cuda")

            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=250,     # ✅ 200 → 250: дэлгэрэнгүй caption
                do_sample=True,
                temperature=0.6,        # ✅ 0.7 → 0.6: тогтвортой гаралт
                top_p=0.9
            )

            generated_ids_trimmed = [
                out_ids[len(in_ids):]
                for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            caption = self.processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )[0]

            return caption.strip()

        except Exception as e:
            print(f"⚠️  Алдаа гарлаа ({image_path}): {e}")
            return None

    def process_dataset(
        self,
        image_dir: str,
        output_dir: str,
        trigger_word: str = "sks person"
    ) -> dict:
        """
        Хавтас дахь бүх зургийг боловсруулж caption үүсгэн
        train_text_to_image.py-д шаардагдах форматаар хадгалах.

        Args:
            image_dir: Оролтын зургийн хавтас
            output_dir: Гаралтын хавтас
            trigger_word: Fine-tuning-д ашиглах trigger үг
        """
        image_dir = Path(image_dir)
        output_dir = Path(output_dir)
        train_dir = output_dir / "train"
        train_dir.mkdir(exist_ok=True, parents=True)

        image_files = []
        for ext in ['.jpg', '.jpeg', '.png', '.webp']:
            image_files.extend(list(image_dir.glob(f'*{ext}')))

        print(f"📂 Олдсон зургийн тоо: {len(image_files)}")

        captions = {}
        metadata_lines = []

        for img_path in tqdm(image_files, desc="Caption үүсгэж байна"):
            caption = self.generate_caption(str(img_path))
            if caption:
                # ✅ Trigger word нэмэх — fine-tuning-д чухал
                full_caption = f"{trigger_word}, {caption}"
                captions[img_path.name] = full_caption

                # train/ хавтасанд зураг + caption хуулах
                dest_img = train_dir / img_path.name
                shutil.copy2(img_path, dest_img)

                # .txt caption файл үүсгэх
                txt_path = train_dir / (img_path.stem + ".txt")
                txt_path.write_text(full_caption, encoding="utf-8")

                # metadata.jsonl мөр нэмэх
                metadata_lines.append(json.dumps({
                    "file_name": img_path.name,
                    "text": full_caption
                }, ensure_ascii=False))

        # captions.json хадгалах
        captions_file = output_dir / "captions.json"
        with open(captions_file, 'w', encoding='utf-8') as f:
            json.dump(captions, f, indent=2, ensure_ascii=False)

        # metadata.jsonl хадгалах (HuggingFace формат)
        metadata_file = train_dir / "metadata.jsonl"
        with open(metadata_file, 'w', encoding='utf-8') as f:
            f.write("\n".join(metadata_lines))

        # dataset_info.json хадгалах
        info = {
            "dataset_info": {
                "total_images": len(captions),
                "trigger_word": trigger_word,
                "caption_file": str(captions_file),
                "train_dir": str(train_dir),
                "image_dir": str(image_dir)
            }
        }
        with open(output_dir / "dataset_info.json", 'w') as f:
            json.dump(info, f, indent=2)

        print(f"\n✅ {len(captions)} caption үүсгэж хадгалагдлаа!")
        print(f"   Caption файл: {captions_file}")
        print(f"   Train хавтас: {train_dir}")
        print(f"   Trigger word: '{trigger_word}'")
        return captions


# ── Ажиллуулах ──────────────────────────────────────────────────────────
IMAGE_DIR  = "/content/my_images"   # Өөрийн зургийн хавтас
OUTPUT_DIR = "/content/my_dataset"  # Гаралтын хавтас
TRIGGER    = "sks person"           # Fine-tuning trigger үг

captioner = ImageCaptioner(model_path="/workspace/Qwen2-VL-7B")
captions = captioner.process_dataset(IMAGE_DIR, OUTPUT_DIR, trigger_word=TRIGGER)

# Caption-уудын жишээ харуулах
print("\n📋 Caption-уудын жишээ:")
for name, cap in list(captions.items())[:3]:
    print(f"  [{name}]: {cap[:120]}...")


---
## 3-р алхам: Загварыг Fine-Tuning хийх

### Fine-Tuning параметрүүд

| Параметр | Утга | Тайлбар |
|----------|------|---------|
| `--resolution` | 512 | Сургалтын зургийн хэмжээ |
| `--train_batch_size` | 1 | T4 GPU-д нэгээс илүү batch багтахгүй |
| `--gradient_accumulation_steps` | 4 | 4 batch хуримтлуулаад нэг update |
| `--mixed_precision` | fp16 | Float16 ашиглан 2× хурдасгах |
| `--use_8bit_adam` | ✅ | Adam-ийн санах ойг 75% бууруулна |
| `--gradient_checkpointing` | ✅ | Gradient санах ойг бууруулна |
| `--max_train_steps` | 300 | ✅ 50→300: загвар онцлогийг бүрэн сурна |
| `--lr_scheduler` | cosine | ✅ constant→cosine: тогтворжиж хурд буурна |

> ⚠️ **T4 GPU (15GB) дээр `--use_8bit_adam` заавал шаардлагатай!**

> ⏳ ~20-30 минут үргэлжилнэ (300 step).


In [ ]:
# Diffusers репозитор татах (train скрипт авахад)
!git clone https://github.com/huggingface/diffusers.git
print("✅ diffusers репозитор татагдлаа!")


In [ ]:
%%capture
%env MODEL_NAME=runwayml/stable-diffusion-v1-5
%env DATASET_DIR=/content/my_dataset/train
%env OUTPUT_DIR=/content/moon-astronaut-model
%env MAX_STEPS=300
%env TRIGGER=sks person


In [ ]:
!accelerate launch diffusers/examples/text_to_image/train_text_to_image.py \
  --pretrained_model_name_or_path=$MODEL_NAME \
  --train_data_dir=$DATASET_DIR \
  --caption_column="text" \
  --use_ema \
  --use_8bit_adam \
  --resolution=512 --center_crop --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --mixed_precision="fp16" \
  --max_train_steps=$MAX_STEPS \
  --learning_rate=5e-05 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=50 \
  --max_grad_norm=1 \
  --output_dir=$OUTPUT_DIR \
  --validation_prompt="sks person in white NASA spacesuit on lunar surface, photorealistic" \
  --validation_epochs=50 \
  --checkpointing_steps=100

print("\n✅ Fine-tuning дууслаа! Загвар '/content/moon-astronaut-model' хавтасанд хадгалагдлаа.")


---
## 4-р алхам: Fine-tuned загвараас зураг авах ба харьцуулах

### 4а. Fine-tuned загварыг ачаалах


In [ ]:
import torch
from diffusers import StableDiffusionPipeline

print("⬇️  Fine-tuned загварыг ачааллаж байна...")

pipe_ft = StableDiffusionPipeline.from_pretrained(
    "/content/moon-astronaut-model",
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe_ft = pipe_ft.to("cuda")
pipe_ft.enable_attention_slicing()

print("✅ Fine-tuned загвар ачаалагдлаа!")


### 4б. Fine-tuned загвараас зураг үүсгэх

`sks person` trigger word ашиглан загвар өөрийн онцлогийг санана.


In [ ]:
import torch

# ✅ Сайжруулсан prompt: trigger word + нарийвчилсан тайлбар
FT_PROMPT = (
    "sks person in white NASA spacesuit standing on grey lunar surface, "
    "Earth visible in black starry sky, cinematic 4k, sharp focus, "
    "photorealistic, dramatic side lighting, helmet visor reflection"
)
NEGATIVE_PROMPT = (
    "blurry, low quality, deformed, extra limbs, bad anatomy, "
    "text, watermark, cartoon, painting, nsfw"
)

print(f"🎨 Prompt: '{FT_PROMPT}'")
print("⏳ Fine-tuned загвараас 4 зураг үүсгэж байна...")

ft_images = []
for i in range(4):
    print(f"  {i+1}/4 үүсгэж байна...")
    torch.cuda.empty_cache()

    with torch.no_grad():
        image = pipe_ft(
            FT_PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=30,
            guidance_scale=9.0,
            height=512,
            width=512,
        ).images[0]
    ft_images.append(image)

print(f"\n✅ {len(ft_images)} зураг үүслээ!")


In [ ]:
print("🖼️  Fine-tuned загвараас гарсан зургууд:")
plot_images(ft_images, titles=[f"Fine-tuned #{i+1}" for i in range(len(ft_images))])


### 4в. Img2Img — Өөрийн зургаар нүүрний онцлог хадгалах

`StableDiffusionImg2ImgPipeline` ашиглан өөрийн зургийг оролт болгон өгч,
`strength` параметраар хэр их өөрчлөхийг тохируулна:

| strength | Утга |
|----------|------|
| 0.4–0.5 | Нүүрний онцлог маш сайн хадгалагдана |
| 0.6–0.75 | Тэнцвэртэй: нүүр хадгалагдаж, стиль өөрчлөгдөнө |
| 0.8–0.95 | Бүрэн өөрчлөгдөнө (нүүр алдагдаж болно) |


In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

# Fine-tuned загвараас img2img pipeline үүсгэх (нэг загвар, хоёр pipeline)
pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
    "/content/moon-astronaut-model",
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe_img2img = pipe_img2img.to("cuda")
pipe_img2img.enable_attention_slicing()

print("✅ Img2Img pipeline бэлэн боллоо!")


In [ ]:
import torch
from PIL import Image
from google.colab import files

# ── Өөрийн зургийг upload хийх ──────────────────────────────────────────
print("📸 Өөрийн зургийг upload хийна уу:")
uploaded_photo = files.upload()

# Upload хийсэн зургийг авах
photo_filename = list(uploaded_photo.keys())[0]
init_image = Image.open(photo_filename).convert("RGB")

# 512×512 болгон resize (fine-tuning-д ашигласан хэмжээтэй нийцүүлэх)
init_image = init_image.resize((512, 512), Image.LANCZOS)

plt.figure(figsize=(4, 4))
plt.imshow(init_image)
plt.axis('off')
plt.title("Оролтын зураг (өөрийн фото)")
plt.show()
print(f"✅ '{photo_filename}' зураг бэлэн болсон: {init_image.size}")


In [ ]:
import torch

IMG2IMG_PROMPT = (
    "sks person in white NASA spacesuit standing on grey lunar surface, "
    "Earth visible in black starry sky, cinematic 4k, sharp focus, "
    "photorealistic, helmet visor, dramatic lighting"
)
NEGATIVE_PROMPT = (
    "blurry, low quality, deformed, extra limbs, bad anatomy, "
    "text, watermark, cartoon, extra face, disfigured"
)

# strength туршилт — өөрийн зургийн хадгалалт vs стиль
STRENGTH_VALUES = [0.5, 0.65, 0.75, 0.85]

print(f"🎨 Prompt: '{IMG2IMG_PROMPT}'")
print(f"⏳ {len(STRENGTH_VALUES)} өөр strength-ээр зураг үүсгэж байна...")

img2img_results = []

for strength in STRENGTH_VALUES:
    print(f"  strength={strength} үүсгэж байна...")
    torch.cuda.empty_cache()

    with torch.no_grad():
        result = pipe_img2img(
            prompt=IMG2IMG_PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=init_image,
            strength=strength,          # Хэр их өөрчлөх
            num_inference_steps=30,
            guidance_scale=9.0,
        ).images[0]
    img2img_results.append(result)

print(f"\n✅ {len(img2img_results)} зураг үүслээ!")


In [ ]:
import matplotlib.pyplot as plt

# Оролтын зураг + img2img үр дүнг харьцуулан харуулах
n = len(img2img_results) + 1
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

axes[0].imshow(init_image)
axes[0].axis('off')
axes[0].set_title("Оролт (өөрийн фото)", fontsize=10)

for i, (ax, img, s) in enumerate(zip(axes[1:], img2img_results, STRENGTH_VALUES)):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"strength={s}", fontsize=10)

plt.suptitle("Img2Img: Өөрийн зургийг сарны гадаргуу дээр", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("✅ Харьцуулалт дууслаа! Хамгийн тохирох strength-ийг сонгоно уу.")


### 4г. Sanity check — Санах ой цэвэрлэх


In [ ]:
print("🧹 GPU санах ойг цэвэрлэж байна...")

del pipe_ft
del pipe_img2img
del ft_images
del img2img_results
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    total_mem = torch.cuda.mem_get_info()[1] / 1e9
    print(f"✅ GPU санах ой чөлөөлөгдлөө: {free_mem:.1f}/{total_mem:.1f} GB чөлөөтэй")


---
## 📊 Дүгнэлт ба харьцуулалт

### Суурь vs Fine-tuned vs Img2Img

| | Суурь загвар | Fine-tuned | Img2Img (Fine-tuned) |
|---|---|---|---|
| Загвар | SD1.5 | SD1.5 + өөрийн зураг | SD1.5 + өөрийн зураг |
| Нүүрний онцлог | ❌ Хадгалагдахгүй | 🔶 Хэсэгчлэн | ✅ Хадгалагдана |
| Prompt дагах | ✅ | ✅ | ✅ |
| Ашиглахад хялбар | ✅ | ✅ | 🔶 Зураг upload шаардлагатай |

### Зөвлөмжүүд

1. **strength=0.6–0.7** нь ихэвчлэн хамгийн тэнцвэртэй үр дүн өгнө
2. **guidance_scale=8–10** нь prompt-г чанд дагуулна
3. **num_inference_steps=28–35** нь чанар ба хурдны сайн тэнцвэр
4. Trigger word `sks person`-г prompt-д заавал оруулах
5. Нарийвчилсан negative_prompt нэмснээр гажуудал буурна
